<a href="https://colab.research.google.com/github/JenniEun/pose_extraction_mediapipe/blob/main/Classification%20using%20transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing the libraries

In [1]:
!pip install torch torchvision torchaudio
!pip install transformers
!pip install scikit-learn
!pip install rouge-score nltk


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=228c4bf4d015a708961d9156660285d55675363cd5633cd6416b1fb8c807a6bd
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


Upload the Dataset

In [2]:
from google.colab import files
uploaded = files.upload()  # Upload ASL_landmarks.csv and asl_test_landmarks.csv


Saving ASL_landmarks.csv to ASL_landmarks.csv
Saving asl_test_landmarks.csv to asl_test_landmarks.csv


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load CSVs
train_df = pd.read_csv('ASL_landmarks.csv')
test_df = pd.read_csv('asl_test_landmarks.csv')




In [7]:
all_labels = np.concatenate([train_df['label'].values, test_df['label'].values])
le = LabelEncoder()
le.fit(all_labels)
y_train = le.transform(train_df['label'].values)
y_test = le.transform(test_df['label'].values)


Create a pytorch dataset and data loader:
#this Wraps your CSV landmark data into a PyTorch Dataset.

#Converts features to torch.float32 and labels to torch.long.

#DataLoader handles batching and shuffling during training.

In [11]:
import pandas as pd
import numpy as np

# Load CSVs
train_df = pd.read_csv('ASL_landmarks.csv')
test_df = pd.read_csv('asl_test_landmarks.csv')

# Original Error Explanation:
# The `KeyError: "['x21', 'y21', 'z21'] not in index"` occurs because the list of
# `feature_cols` was generated from the columns of `train_df` (which includes 'x21', 'y21', 'z21'
# in its header). However, `test_df` (from 'asl_test_landmarks.csv') does not have these specific
# columns; instead, it appears to have landmark data indexed from 'x0' to 'x20'.
#
# Additionally, `train_df` in the current kernel state is an `Empty DataFrame`,
# suggesting that 'ASL_landmarks.csv' might only contain a header row or is malformed,
# leading to no actual training data being loaded.

# Fix for the KeyError:
# We need to ensure that `feature_cols` accurately reflects the columns present
# in the DataFrame being processed. Since `test_df` is causing the KeyError,
# we will derive its feature columns directly from `test_df.columns`.
# This will exclude any columns like 'x21', 'y21', 'z21' that are not actually in `test_df`.

# Determine feature columns for test_df.
# This assumes 'label' is the only non-feature column in test_df.
feature_cols_for_test_df = [c for c in test_df.columns if c != 'label']

# Now, we need to handle train_df. Since it's empty and has different columns (x1-x21 vs x0-x20),
# directly using `feature_cols_for_test_df` on `train_df` would likely cause a new KeyError
# (e.g., 'x0' not in train_df).
# To prevent this and acknowledge the empty train_df, we handle it separately.

# Define feature columns for train_df based on its header.
# Note: If train_df is empty, this list will represent its header, but it won't have data.
feature_cols_for_train_df = [c for c in train_df.columns if c != 'label']

# Convert all feature columns to numeric, coerce errors to NaN
# Apply to train_df using its specific feature columns.
# This block is made robust to handle an empty train_df.
if not train_df.empty and len(feature_cols_for_train_df) > 0:
    train_df[feature_cols_for_train_df] = train_df[feature_cols_for_train_df].apply(pd.to_numeric, errors='coerce')
    train_df = train_df.dropna(subset=feature_cols_for_train_df)
elif not train_df.empty and len(feature_cols_for_train_df) == 0: # Case where only 'label' column or no columns
    pass # No feature columns to process
else: # train_df is empty
    train_df = pd.DataFrame(columns=feature_cols_for_train_df + ['label'] if 'label' in train_df.columns else feature_cols_for_train_df) # Create an empty df with expected columns

# Apply to test_df using its specific feature columns (which resolved the KeyError).
test_df[feature_cols_for_test_df] = test_df[feature_cols_for_test_df].apply(pd.to_numeric, errors='coerce')
test_df = test_df.dropna(subset=feature_cols_for_test_df)

print(f"Cleaned training samples: {len(train_df)}, cleaned test samples: {len(test_df)}")

# Separate features and labels
# Handle X_train and y_train carefully if train_df is empty
if not train_df.empty:
    X_train = train_df[feature_cols_for_train_df].values.astype(np.float32)
    y_train = train_df['label'].values
else:
    # If train_df is empty, create empty numpy arrays with appropriate shapes.
    # The shape of X_train needs to match the number of feature columns.
    X_train = np.array([], dtype=np.float32).reshape(0, len(feature_cols_for_train_df))
    y_train = np.array([], dtype=object) # labels might be strings/objects

X_test = test_df[feature_cols_for_test_df].values.astype(np.float32)
y_test = test_df['label'].values

Cleaned training samples: 0, cleaned test samples: 14
